# Vision-Driven Deep Reinforcement Learning for Dynamic Warehouse Task and Robot Allocation
### IEEE 2nd International Conference on Logistics (ICL 2026)
**Author:** Atta | **Field:** Optimization Algorithms

**Primary dataset:** [Warehouse Operation Dataset](https://www.kaggle.com/datasets/srvydv1234567/warehouse-operation-dataset)

This notebook implements a reproducible closed-loop benchmark: a congestion-aware 8x8 grid warehouse simulator calibrated from warehouse-operation data, with WMS-only heuristics compared against vision-augmented RL under shared perception noise. Downstream logistics KPIs include throughput, completion time, travel distance, battery use, near-collisions, and priority fairness.


In [1]:

# CELL 1 - SETUP
import os, sys, random, warnings, zipfile, glob, time, copy
from collections import deque, defaultdict

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.lines import Line2D
import seaborn as sns
from scipy.optimize import linear_sum_assignment
from scipy.stats import mannwhitneyu

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.4f}'.format)

SEED = 42
EVAL_SEEDS = [42, 123, 456]
N_TRAIN = 600
N_EVAL = 50

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed(s)

set_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

OUT = '/kaggle/working'
FIG_DIR = f'{OUT}/figures'
TAB_DIR = f'{OUT}/tables'
for d in [FIG_DIR, TAB_DIR]: os.makedirs(d, exist_ok=True)

plt.rcParams.update({
    'figure.dpi':200,'font.family':'DejaVu Serif','font.size':11,
    'axes.titlesize':12,'axes.titleweight':'bold','axes.labelsize':11,
    'legend.fontsize':9,'xtick.labelsize':9,'ytick.labelsize':9,
    'axes.grid':True,'grid.alpha':0.3,'grid.linestyle':'--',
    'axes.spines.top':False,'axes.spines.right':False,
    'figure.constrained_layout.use':True,
})

ALGO_COLORS = {
    'Random':'#e74c3c','Nearest-WMS':'#3498db','Hungarian-WMS':'#2ecc71',
    'Genetic-WMS':'#1abc9c','Q-Learning':'#f39c12','DQN-Vision':'#9b59b6',
    'Constrained-DQN':'#8e44ad',
}
ALGO_MARKERS = {'Random':'o','Nearest-WMS':'s','Hungarian-WMS':'^','Genetic-WMS':'v',
                'Q-Learning':'D','DQN-Vision':'*','Constrained-DQN':'P'}
ALGOS = list(ALGO_COLORS.keys())

def save_fig(name):
    p = f'{FIG_DIR}/{name}.pdf'
    plt.savefig(p, format='pdf', bbox_inches='tight', dpi=200)
    plt.close('all')
    print('  Saved figure:', p)

def smooth(arr, w=20):
    arr = np.array(arr, dtype=float)
    if len(arr) < w: return arr
    pad = np.pad(arr, (w//2, w//2), mode='reflect')
    return np.convolve(pad, np.ones(w)/w, mode='valid')[:len(arr)]

print('Setup complete.')


Device: cuda
Setup complete.


In [2]:

# CELL 2 - ROBUST DATASET LOADING
DATA_SOURCE = 'unknown'

def _read_table(fp):
    ext = fp.lower().split('.')[-1]
    if ext in ('csv','txt','tsv'):
        sep = '\t' if ext == 'tsv' else ','
        return pd.read_csv(fp, sep=sep, low_memory=False)
    if ext == 'xlsx':
        return pd.read_excel(fp)
    if ext == 'parquet':
        return pd.read_parquet(fp)
    return None

def load_warehouse_data():
    global DATA_SOURCE
    paths = []
    for base in glob.glob('/kaggle/input/*'):
        if os.path.isdir(base):
            for root, _, files in os.walk(base):
                for fn in files:
                    if fn.lower().endswith(('.csv','.txt','.tsv','.xlsx','.parquet')):
                        paths.append(os.path.join(root, fn))
    for p in ['warehouse_operation.csv','data/warehouse_operation.csv']:
        if os.path.isfile(p): paths.append(p)

    print('Candidate data files:', len(paths))
    for fp in paths[:15]:
        print(' ', fp, f'({os.path.getsize(fp)//1024} KB)')

    dfs = {}
    for fp in paths:
        try:
            df = _read_table(fp)
            if df is not None and len(df) >= 50 and df.shape[1] >= 5:
                dfs[fp] = df
        except Exception as e:
            print('  skip', fp, e)

    if dfs:
        best = max(dfs, key=lambda k: dfs[k].shape[0]*dfs[k].shape[1])
        DATA_SOURCE = 'kaggle:' + os.path.basename(best)
        print('Loaded Kaggle file:', best, dfs[best].shape)
        return dfs[best].copy()

    DATA_SOURCE = 'bundled_fallback'
    print('No Kaggle CSV found - using bundled warehouse-operation fallback (deterministic).')
    rng = np.random.RandomState(SEED)
    n = 1000
    return pd.DataFrame({
        'forklifts': rng.uniform(0,1,n),
        'per_unload_docks': rng.uniform(0,1,n),
        'per_order_assembling': rng.uniform(0,1,n),
        'unloading_trucks_per_hour': rng.uniform(0,1,n),
        'loading_trucks_per_hour': rng.uniform(0,1,n),
        'orders_per_hour': rng.uniform(0,1,n),
        'trucks_capacity': rng.uniform(0,1,n),
        'minimum_order_size': rng.randint(0,2,n),
        'maximum_order_size': rng.uniform(0,1,n),
        'orders_per_forklift': rng.exponential(2,n),
        'truck_efficiency': rng.exponential(3,n),
        'order_size_ratio': rng.gamma(2,1,n),
    })

df_raw = load_warehouse_data()
print('DATA_SOURCE =', DATA_SOURCE)
print('Shape:', df_raw.shape)
print(df_raw.describe().to_string())


Candidate data files: 1
  /kaggle/input/datasets/srvydv1234567/warehouse-operation-dataset/Updated_Warehouse_Operations_Dataset.csv (54 KB)
Loaded Kaggle file: /kaggle/input/datasets/srvydv1234567/warehouse-operation-dataset/Updated_Warehouse_Operations_Dataset.csv (1000, 12)
DATA_SOURCE = kaggle:Updated_Warehouse_Operations_Dataset.csv
Shape: (1000, 12)
       forklifts  Per unload docks  Per order assembling  Unloading trucks per hour  Loading trucks per hour  orders per hour  Trucks capacity  Minimum order size  Maximum order size  orders_per_forklift  truck_efficiency  order_size_ratio
count  1000.0000         1000.0000             1000.0000                  1000.0000                1000.0000        1000.0000        1000.0000           1000.0000           1000.0000            1000.0000         1000.0000         1000.0000
mean      0.4842            0.4887                0.4956                     0.5024                   0.4956           0.4950           0.5009              0.4910 

In [3]:

# CELL 3 - FIG 1 EDA
num_cols = [c for c in df_raw.select_dtypes(include=[np.number]).columns
            if not any(x in c.lower() for x in ['id','index'])][:6]
fig = plt.figure(figsize=(16,10))
fig.suptitle('Fig. 1: Warehouse Operation Dataset - Exploratory Data Analysis', fontsize=14, fontweight='bold')
gs = gridspec.GridSpec(3,4,figure=fig,hspace=0.58,wspace=0.42)
for idx,col in enumerate(num_cols):
    row,c = divmod(idx,3)
    ax = fig.add_subplot(gs[row,c])
    data = df_raw[col].dropna()
    ax.hist(data,bins=30,color='#3498db',alpha=0.75,edgecolor='white',lw=0.5)
    mu,sd = data.mean(),data.std()
    ax.axvline(mu,color='#e74c3c',ls='--',lw=1.5,label=f'mu={mu:.2f}')
    label = col.replace('_',' ').title()
    ax.set_title(f'{label}',fontsize=9); ax.legend(fontsize=7)
ax_corr = fig.add_subplot(gs[2,:2])
sns.heatmap(df_raw[num_cols].corr(),ax=ax_corr,annot=True,fmt='.2f',cmap='coolwarm',center=0,vmin=-1,vmax=1,linewidths=0.5,annot_kws={'size':7})
ax_corr.set_title('Correlation Matrix',fontweight='bold')
ax_pie = fig.add_subplot(gs[2,2:])
if 'orders_per_hour' in df_raw.columns:
    q = pd.qcut(df_raw['orders_per_hour'],3,labels=['Low','Med','High'],duplicates='drop')
    vc = q.value_counts()
    ax_pie.pie(vc,labels=vc.index,autopct='%1.1f%%',startangle=90,textprops={'fontsize':9})
    ax_pie.set_title('Order-Rate Terciles',fontweight='bold')
save_fig('fig_01_dataset_eda')
print('Figure 1 saved. Source:', DATA_SOURCE)


  Saved figure: /kaggle/working/figures/fig_01_dataset_eda.pdf
Figure 1 saved. Source: kaggle:Updated_Warehouse_Operations_Dataset.csv


In [4]:

# CELL 4 - SIMULATOR CALIBRATION (same df_raw as Cell 2)
def _col_mean(df, names, default):
    for n in names:
        if n in df.columns:
            v = pd.to_numeric(df[n], errors='coerce').dropna()
            if len(v): return float(v.mean())
    return default

orders_h = _col_mean(df_raw, ['orders_per_hour','order_rate'], 0.5)
asm      = _col_mean(df_raw, ['per_order_assembling','assembly_time_min'], 0.5)
fork     = _col_mean(df_raw, ['forklifts','forklift_count'], 0.5)
dist_p   = _col_mean(df_raw, ['order_size_ratio','distance_m'], 1.0)

TASK_ARRIVAL_RATE = float(np.clip(orders_h * 0.25, 0.06, 0.35))
SERVICE_STEPS     = int(np.clip(2 + asm * 6, 2, 8))
AVG_DIST          = int(np.clip(2 + dist_p * 2, 2, 10))
CONGESTION_PROB   = float(np.clip(0.08 + fork * 0.18, 0.08, 0.28))
PRIORITY_WEIGHTS  = [0.50, 0.35, 0.15]

print('=== Calibrated Parameters (from', DATA_SOURCE, ') ===')
print('  Arrival rate   :', round(TASK_ARRIVAL_RATE,3))
print('  Service steps  :', SERVICE_STEPS)
print('  Congestion prob:', round(CONGESTION_PROB,3))
print('  Priority weights:', PRIORITY_WEIGHTS)

rows = []
for col in df_raw.select_dtypes(include=[np.number]).columns[:12]:
    s = df_raw[col].describe()
    rows.append({'Variable':col,'N':int(s['count']),'Mean':round(s['mean'],4),
                 'Std':round(s['std'],4),'Min':round(s['min'],4),
                 'Median':round(s['50%'],4),'Max':round(s['max'],4)})
df_t1 = pd.DataFrame(rows)
df_t1.to_csv(f'{TAB_DIR}/table_01_dataset_summary.csv', index=False)
print('Table 1 saved from current df_raw.')
print(df_t1.to_string(index=False))


=== Calibrated Parameters (from kaggle:Updated_Warehouse_Operations_Dataset.csv ) ===
  Arrival rate   : 0.125
  Service steps  : 5
  Congestion prob: 0.167
  Priority weights: [0.5, 0.35, 0.15]
Table 1 saved from current df_raw.
                 Variable    N      Mean        Std    Min  Median         Max
                forklifts 1000    0.4842     0.3009 0.0000  0.4706      1.0000
         Per unload docks 1000    0.4887     0.3113 0.0000  0.5000      1.0000
     Per order assembling 1000    0.4956     0.3066 0.0000  0.5000      1.0000
Unloading trucks per hour 1000    0.5024     0.3127 0.0000  0.5000      1.0000
  Loading trucks per hour 1000    0.4956     0.3160 0.0000  0.5000      1.0000
          orders per hour 1000    0.4950     0.3189 0.0000  0.5000      1.0000
          Trucks capacity 1000    0.5008     0.2964 0.0000  0.5000      1.0000
       Minimum order size 1000    0.4910     0.5002 0.0000  0.0000      1.0000
       Maximum order size 1000    0.4858     0.3382 0.0000 

In [5]:

# CELL 5 - CONGESTION-AWARE WAREHOUSE ENVIRONMENT
class WarehouseEnv:
    GRID=8; N_ROBOTS=3; N_TASKS=6; MAX_STEPS=150
    BATT_MAX=100.0; BATT_DRAIN=0.4; COLL_RAD=1
    R_TASK=10.0; R_STEP=-0.02; R_BATT_LOW=-1.5; R_COLL=-3.0
    R_PRIO=[0.0,1.5,3.0]

    def __init__(self, noise_sigma=0.0, use_vision=True, use_priority=True,
                 planner_mode='wms', service_steps=None, collision_penalty=None,
                 priority_weights=None, congestion_prob=None):
        self.noise_sigma = noise_sigma
        self.use_vision = use_vision
        self.use_priority = use_priority
        self.planner_mode = planner_mode
        self.service_steps = service_steps or SERVICE_STEPS
        self.R_COLL = collision_penalty if collision_penalty is not None else self.R_COLL
        self.priority_weights = priority_weights or PRIORITY_WEIGHTS
        self.congestion_prob = congestion_prob if congestion_prob is not None else CONGESTION_PROB
        self.shelf = np.zeros((self.GRID,self.GRID),dtype=np.int8)
        for row in [1,3,5,7]:
            for col in range(1,7): self.shelf[row,col]=1
        self.aisle=[(r,c) for r in range(self.GRID) for c in range(self.GRID) if self.shelf[r,c]==0]
        self.spawns=[(0,0),(2,0),(4,0)]
        self.dynamic_blocks=set()
        self.state_dim = self.N_ROBOTS*4 + self.N_TASKS*5 + (16 if use_vision else 0) + self.N_ROBOTS
        self.action_dim = self.N_TASKS
        self.reset()

    def is_blocked(self,r,c):
        return self.shelf[r,c]==1 or (r,c) in self.dynamic_blocks

    def _bfs_dist(self,start,goal):
        if start==goal: return 0
        q=deque([(start,0)]); seen={start}
        while q:
            (r,c),d=q.popleft()
            for dr,dc in [(-1,0),(1,0),(0,-1),(0,1)]:
                nr,nc=r+dr,c+dc
                if not(0<=nr<self.GRID and 0<=nc<self.GRID): continue
                if self.is_blocked(nr,nc): continue
                if (nr,nc)==goal: return d+1
                if (nr,nc) not in seen:
                    seen.add((nr,nc)); q.append(((nr,nc),d+1))
        return self.GRID*3

    def _bfs_next(self,start,goal):
        if start==goal: return start
        q=deque([start]); seen={start}; parent={start:None}
        while q:
            cur=q.popleft()
            if cur==goal:
                node=goal
                while parent[node] is not None and parent[node]!=start:
                    node=parent[node]
                return node
            r,c=cur
            for dr,dc in [(-1,0),(1,0),(0,-1),(0,1)]:
                nr,nc=r+dr,c+dc; nxt=(nr,nc)
                if not(0<=nr<self.GRID and 0<=nc<self.GRID): continue
                if self.is_blocked(nr,nc): continue
                if nxt not in seen:
                    seen.add(nxt); parent[nxt]=cur; q.append(nxt)
        return start

    def _update_dynamic_blocks(self):
        if random.random()<self.congestion_prob:
            free=[c for c in self.aisle if c not in self.dynamic_blocks and tuple(c) not in map(tuple,self.rpos)]
            if free: self.dynamic_blocks.add(random.choice(free))
        if self.dynamic_blocks and random.random()<0.10:
            self.dynamic_blocks.remove(random.choice(list(self.dynamic_blocks)))

    def _occ4(self):
        occ=self.shelf.astype(np.float32).copy()
        for r,c in self.dynamic_blocks: occ[r,c]=0.85
        for rp in self.rpos: occ[rp[0],rp[1]]=0.5
        if self.noise_sigma>0:
            occ=np.clip(occ+np.random.normal(0,self.noise_sigma,occ.shape),0,1)
        return occ.reshape(4,2,4,2).mean(axis=(1,3)).flatten()

    def perceived_task_pos(self,j):
        pos=np.array(self.tpos[j],dtype=float)
        if self.noise_sigma>0:
            pos+=np.random.normal(0,self.noise_sigma*self.GRID*0.35,2)
        return np.clip(np.round(pos),0,self.GRID-1).astype(int).tolist()

    def _mdist(self,a,b):
        return abs(a[0]-b[0])+abs(a[1]-b[1])

    def _task_cost(self,robot_idx,task_idx):
        start=tuple(self.rpos[robot_idx]); goal=tuple(self.tpos[task_idx])
        if self.planner_mode=='vision':
            return self._bfs_dist(start,goal)
        return self._mdist(start,tuple(self.perceived_task_pos(task_idx)))

    def _state(self,robot_idx):
        g=float(self.GRID); s=[]
        for i in range(self.N_ROBOTS):
            s += [self.rpos[i][0]/g,self.rpos[i][1]/g,self.rbatt[i]/self.BATT_MAX,float(self.rbusy[i])]
        for j in range(self.N_TASKS):
            pr=self.tprio[j]/3.0 if self.use_priority else 0.333
            av=float(self.tactive[j] and not self.tassigned[j])
            pt=self.perceived_task_pos(j)
            bd=self._bfs_dist(tuple(self.rpos[robot_idx]),tuple(self.tpos[j]))/(self.GRID*3)
            s += [pt[0]/g,pt[1]/g,pr,av,bd]
        if self.use_vision: s += self._occ4().tolist()
        oh=[0.0]*self.N_ROBOTS; oh[robot_idx]=1.0; s += oh
        return np.array(s,dtype=np.float32)

    def action_mask(self):
        m=np.zeros(self.N_TASKS,dtype=np.float32)
        for j in self.avail_tasks(): m[j]=1.0
        return m

    def reset(self):
        self.t=0; self.ep_tasks=0; self.ep_colls=0; self.ep_dist=0.0; self.ep_batt=0.0
        self.ep_rew=0.0; self.ep_ctimes=[]; self.ep_prio_waits={1:[],2:[],3:[]}
        self.dynamic_blocks=set()
        self.rpos=[list(self.spawns[i]) for i in range(self.N_ROBOTS)]
        self.rbatt=[self.BATT_MAX]*self.N_ROBOTS; self.rbusy=[False]*self.N_ROBOTS
        self.rtarget=[None]*self.N_ROBOTS; self.rsvc=[0]*self.N_ROBOTS; self.rtstart=[0]*self.N_ROBOTS
        self.tpos=[[0,0]]*self.N_TASKS; self.tprio=[1]*self.N_TASKS
        self.tactive=[False]*self.N_TASKS; self.tassigned=[False]*self.N_TASKS; self.tspawn=[0]*self.N_TASKS
        for j in range(self.N_TASKS): self._spawn(j)
        return self._state(0)

    def _spawn(self,j):
        occ=set(map(tuple,self.tpos))
        cands=[c for c in self.aisle if c not in occ]
        loc=random.choice(cands if cands else self.aisle)
        self.tpos[j]=list(loc)
        self.tprio[j]=random.choices([1,2,3],weights=self.priority_weights)[0]
        self.tactive[j]=True; self.tassigned[j]=False; self.tspawn[j]=self.t

    def assign(self,robot_idx,task_idx):
        if 0<=task_idx<self.N_TASKS and self.tactive[task_idx] and not self.tassigned[task_idx] and not self.rbusy[robot_idx]:
            self.rtarget[robot_idx]=task_idx; self.rbusy[robot_idx]=True
            self.tassigned[task_idx]=True; self.rtstart[robot_idx]=self.t
            return True
        return False

    def _move(self,i):
        if not self.rbusy[i] or self.rtarget[i] is None: return
        j=self.rtarget[i]; goal=tuple(self.tpos[j]); pos=tuple(self.rpos[i])
        if pos==goal: return
        nxt=self._bfs_next(pos,goal)
        if nxt!=pos:
            self.ep_dist += self._mdist(pos,nxt); self.ep_batt += self.BATT_DRAIN
            self.rpos[i]=list(nxt); self.rbatt[i]=max(0.0,self.rbatt[i]-self.BATT_DRAIN)

    def step(self,assignments=None):
        if assignments:
            for ri,ti in assignments.items(): self.assign(ri,ti)
        rew=self.R_STEP*self.N_ROBOTS
        for i in range(self.N_ROBOTS):
            if self.rbusy[i] and self.rtarget[i] is not None:
                j=self.rtarget[i]; self._move(i)
                if self.rpos[i]==self.tpos[j]:
                    self.rsvc[i]+=1
                    if self.rsvc[i]>=self.service_steps:
                        ct=self.t-self.rtstart[i]; self.ep_ctimes.append(ct)
                        self.ep_prio_waits[self.tprio[j]].append(ct)
                        self.ep_tasks+=1
                        rew += self.R_TASK + self.R_PRIO[self.tprio[j]-1]
                        self.rbusy[i]=False; self.rtarget[i]=None; self.rsvc[i]=0; self._spawn(j)
            if self.rbatt[i]<15: rew += self.R_BATT_LOW
        for i in range(self.N_ROBOTS):
            for k in range(i+1,self.N_ROBOTS):
                if self._mdist(self.rpos[i],self.rpos[k])<=self.COLL_RAD:
                    self.ep_colls+=1; rew += self.R_COLL
        self._update_dynamic_blocks(); self.ep_rew+=rew; self.t+=1
        done=self.t>=self.MAX_STEPS
        return done

    def avail_tasks(self):
        return [j for j in range(self.N_TASKS) if self.tactive[j] and not self.tassigned[j]]

    def nearest_task(self,robot_idx,exclude=()):
        avail=[j for j in self.avail_tasks() if j not in exclude]
        if not avail: return None
        return avail[int(np.argmin([self._task_cost(robot_idx,j) for j in avail]))]

# smoke test
_e=WarehouseEnv(); _e.reset()
assert _e._state(0).shape==( _e.state_dim,)
print('WarehouseEnv OK | state_dim=', _e.state_dim)
del _e


WarehouseEnv OK | state_dim= 61


In [6]:

# CELL 6 - FIG 2 WAREHOUSE LAYOUT
env_viz=WarehouseEnv(noise_sigma=0.0,use_vision=True); env_viz.reset()
env_viz.dynamic_blocks={(1,2),(3,4),(5,2)}
fig,axes=plt.subplots(1,2,figsize=(14,6))
fig.suptitle('Fig. 2: Grid Warehouse with Dynamic Congestion and Noisy Vision Perception',fontsize=13,fontweight='bold')
_cmap=LinearSegmentedColormap.from_list('occ',['#f8f9fa','#f39c12','#495057'])
for ax,noise,title in zip(axes,[0.0,0.18],['(A) True Occupancy','(B) Noisy Perception (sigma=0.18)']):
    G=env_viz.GRID; ax.set_xlim(-0.5,G-0.5); ax.set_ylim(G-0.5,-0.5); ax.set_aspect('equal')
    env_viz.noise_sigma=noise
    occ=env_viz.shelf.astype(float).copy()
    for r,c in env_viz.dynamic_blocks: occ[r,c]=0.85
    for rp in env_viz.rpos: occ[rp[0],rp[1]]=0.5
    if noise>0: occ=np.clip(occ+np.random.RandomState(7).normal(0,noise,occ.shape),0,1)
    ax.imshow(occ,cmap=_cmap,vmin=0,vmax=1,alpha=0.6,extent=[-0.5,G-0.5,G-0.5,-0.5])
    for r in range(G):
        for c in range(G):
            if env_viz.shelf[r,c]:
                ax.add_patch(mpatches.Rectangle((c-0.45,r-0.45),0.9,0.9,fc='#6c757d',ec='#343a40',lw=0.8))
            elif (r,c) in env_viz.dynamic_blocks:
                ax.add_patch(mpatches.Rectangle((c-0.45,r-0.45),0.9,0.9,fc='#e67e22',ec='#d35400',lw=0.8,alpha=0.9))
    for j in range(env_viz.N_TASKS):
        tp=env_viz.tpos[j]; col=['#2ecc71','#f39c12','#e74c3c'][env_viz.tprio[j]-1]
        ax.scatter(tp[1],tp[0],s=100,c=col,marker='D',edgecolors='k',zorder=5)
    for i,rp in enumerate(env_viz.rpos):
        ax.add_patch(plt.Circle((rp[1],rp[0]),0.32,color=['#3498db','#9b59b6','#1abc9c'][i],ec='k',lw=1.2,zorder=6))
        ax.text(rp[1],rp[0],f'R{i+1}',ha='center',va='center',fontsize=7,color='white',fontweight='bold')
    ax.set_title(title,fontweight='bold'); ax.set_xlabel('Column'); ax.set_ylabel('Row')
save_fig('fig_02_warehouse_layout'); del env_viz
print('Figure 2 saved.')


  Saved figure: /kaggle/working/figures/fig_02_warehouse_layout.pdf
Figure 2 saved.


In [7]:

# CELL 7 - METRICS + BASELINE CONTROLLERS
def jains_fairness(waits_by_prio):
    vals=[np.mean(v) for v in waits_by_prio.values() if len(v)>0]
    if not vals: return 1.0
    s=sum(vals); s2=sum(v*v for v in vals)
    return float((s*s)/(len(vals)*s2+1e-9))

def collect_metrics(env):
    return {
        'tasks':env.ep_tasks,'colls':env.ep_colls,'dist':env.ep_dist,'batt':env.ep_batt,
        'ct':float(np.mean(env.ep_ctimes)) if env.ep_ctimes else float(env.MAX_STEPS),
        'reward':env.ep_rew,
        'fairness':jains_fairness(env.ep_prio_waits),
    }

def make_env(**kw):
    return WarehouseEnv(**kw)

def run_episode(env, policy_fn, seed):
    random.seed(seed); np.random.seed(seed); env.reset(); done=False
    while not done:
        idle=[i for i in range(env.N_ROBOTS) if not env.rbusy[i]]
        asgn=policy_fn(env, idle) if idle else {}
        done=env.step(asgn)
    return collect_metrics(env)

def policy_random(env, idle):
    avail=env.avail_tasks(); asgn={}
    for ri in idle:
        if avail:
            ti=random.choice(avail); asgn[ri]=ti; avail=[a for a in avail if a!=ti]
    return asgn

def policy_nearest_wms(env, idle):
    asgn={}; used=set()
    for ri in idle:
        nt=env.nearest_task(ri,exclude=used)
        if nt is not None: asgn[ri]=nt; used.add(nt)
    return asgn

def policy_hungarian_wms(env, idle):
    avail=env.avail_tasks(); asgn={}
    if idle and avail:
        cost=np.full((len(idle),len(avail)),1e6)
        for ri_i,ri in enumerate(idle):
            for ti_i,ti in enumerate(avail):
                cost[ri_i,ti_i]=env._mdist(tuple(env.rpos[ri]),tuple(env.perceived_task_pos(ti)))
        rows,cols=linear_sum_assignment(cost)
        for ri_i,ti_i in zip(rows,cols):
            if cost[ri_i,ti_i]<1e5: asgn[idle[ri_i]]=avail[ti_i]
    return asgn

def policy_genetic_wms(env, idle, samples=48):
    avail=env.avail_tasks()
    if not idle or not avail: return {}
    best=None; best_cost=1e18
    for _ in range(samples):
        tasks=avail.copy(); random.shuffle(tasks)
        asgn={}
        for ri,ti in zip(idle,tasks): asgn[ri]=ti
        cost=sum(env._mdist(tuple(env.rpos[ri]),tuple(env.perceived_task_pos(ti))) for ri,ti in asgn.items())
        if cost<best_cost: best_cost=cost; best=asgn
    return best or {}

POLICIES = {
    'Random': policy_random,
    'Nearest-WMS': policy_nearest_wms,
    'Hungarian-WMS': policy_hungarian_wms,
    'Genetic-WMS': policy_genetic_wms,
}

def eval_policy(name, policy_fn, seeds=EVAL_SEEDS, n_eps=N_EVAL, **env_kw):
    m=defaultdict(list)
    cfg={'planner_mode':'wms', 'use_vision':False}
    cfg.update(env_kw)
    for sd in seeds:
        env=make_env(**cfg)
        for ep in range(n_eps):
            met=run_episode(env, policy_fn, sd*10000+ep)
            for k,v in met.items(): m[k].append(v)
    return dict(m)

print('Baselines ready:', list(POLICIES.keys()))


Baselines ready: ['Random', 'Nearest-WMS', 'Hungarian-WMS', 'Genetic-WMS']


In [8]:

# CELL 8 - Q-LEARNING
class QLAgent:
    def __init__(self,n_actions=6,alpha=0.2,gamma=0.97,eps=0.9,eps_min=0.05,eps_decay=0.995):
        self.n_actions=n_actions; self.alpha=alpha; self.gamma=gamma
        self.eps=eps; self.eps_min=eps_min; self.eps_decay=eps_decay
        self.Q=defaultdict(lambda: np.zeros(n_actions))
    def _key(self,state):
        rx=int(np.clip(state[0]*4,0,3)); ry=int(np.clip(state[1]*4,0,3))
        rb=int(np.clip(state[2]*3,0,2))
        bits=tuple(int(state[12+j*5+3]>0.5) for j in range(6))
        return (rx,ry,rb,sum(bits))
    def act(self,state,mask,greedy=False):
        valid=[i for i in range(self.n_actions) if mask[i]>0]
        if not valid: return 0
        if (not greedy) and random.random()<self.eps:
            return random.choice(valid)
        q=self.Q[self._key(state)]; q=q.copy(); q[mask==0]=-1e9
        a=int(np.argmax(q))
        return a if mask[a]>0 else valid[0]
    def update(self,s,a,r,ns,nd,mask_n):
        k=self._key(s); nk=self._key(ns)
        nq=self.Q[nk].copy(); nq[mask_n==0]=-1e9
        target=r + (0 if nd else self.gamma*np.max(nq))
        self.Q[k][a] += self.alpha*(target-self.Q[k][a])
    def decay(self): self.eps=max(self.eps_min,self.eps*self.eps_decay)

def train_ql(n_train=400, seed_off=0, **env_kw):
    env=make_env(planner_mode='vision', use_vision=True, **env_kw)
    ag=QLAgent(n_actions=env.N_TASKS); rews=[]
    for ep in range(n_train):
        random.seed(SEED+seed_off+ep); env.reset(); done=False; er=0
        while not done:
            idle=[i for i in range(env.N_ROBOTS) if not env.rbusy[i]]
            asgn={}; trans=[]
            for ri in idle:
                s=env._state(ri); mask=env.action_mask()
                a=ag.act(s,mask)
                if mask[a]>0: asgn[ri]=a; trans.append((ri,s,a,mask.copy()))
            done=env.step(asgn)
            for ri,s,a,mask in trans:
                ns=env._state(ri); nmask=env.action_mask()
                ag.update(s,a,env.ep_rew/max(1,len(trans)),ns,done,nmask)
            ag.decay(); er += env.ep_rew
        rews.append(er)
    return ag, rews

print('Q-Learning defined.')


Q-Learning defined.


In [9]:

# CELL 9 - DQN WITH ACTION MASKING
class DQNet(nn.Module):
    def __init__(self,sd,ad):
        super().__init__()
        self.net=nn.Sequential(
            nn.Linear(sd,256),nn.ReLU(),nn.Dropout(0.1),
            nn.Linear(256,128),nn.ReLU(),nn.Linear(128,ad))
    def forward(self,x): return self.net(x)

class ReplayBuffer:
    def __init__(self,cap=30000): self.buf=deque(maxlen=cap)
    def push(self,*x): self.buf.append(x)
    def sample(self,n):
        b=random.sample(self.buf,n)
        s,a,r,ns,d,m,mn=zip(*b)
        return (np.array(s),list(a),np.array(r),np.array(ns),np.array(d),np.array(m),np.array(mn))
    def __len__(self): return len(self.buf)

class DQNAgent:
    def __init__(self,sd,ad,lr=5e-4,gamma=0.97,batch=128,eps=1.0,eps_min=0.05,eps_decay=0.996,tgt=30):
        self.gamma=gamma; self.batch=batch; self.eps=eps; self.eps_min=eps_min
        self.eps_decay=eps_decay; self.tgt=tgt; self.steps=0; self.ad=ad
        self.online=DQNet(sd,ad).to(device); self.target=DQNet(sd,ad).to(device)
        self.target.load_state_dict(self.online.state_dict()); self.target.eval()
        self.opt=optim.Adam(self.online.parameters(),lr=lr)
        self.buf=ReplayBuffer(); self.losses=[]
    def act(self,s,mask,greedy=False):
        valid=np.where(mask>0)[0]
        if len(valid)==0: return 0
        if (not greedy) and random.random()<self.eps:
            return int(random.choice(valid))
        with torch.no_grad():
            q=self.online(torch.FloatTensor(s).unsqueeze(0).to(device)).cpu().numpy()[0]
        q=q.copy(); q[mask==0]=-1e9
        a=int(np.argmax(q)); return a if mask[a]>0 else int(valid[0])
    def learn(self):
        if len(self.buf)<self.batch*4: return
        s,a,r,ns,d,m,mn=self.buf.sample(self.batch)
        s=torch.FloatTensor(s).to(device); a=torch.LongTensor(a).unsqueeze(1).to(device)
        r=torch.FloatTensor(r).unsqueeze(1).to(device); ns=torch.FloatTensor(ns).to(device)
        d=torch.FloatTensor(d).unsqueeze(1).to(device); m=torch.FloatTensor(m).to(device)
        mn=torch.FloatTensor(mn).to(device)
        q=self.online(s).gather(1,a)
        with torch.no_grad():
            qo=self.online(ns); qo=qo.clone(); qo[mn==0]=-1e9
            best=qo.argmax(1,keepdim=True)
            qt=self.target(ns).gather(1,best)
            y=r + self.gamma*qt*(1-d)
        loss=F.smooth_l1_loss(q,y)
        self.opt.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(self.online.parameters(),1.0)
        self.opt.step(); self.losses.append(loss.item()); self.steps+=1
        if self.steps%self.tgt==0: self.target.load_state_dict(self.online.state_dict())
    def decay(self): self.eps=max(self.eps_min,self.eps*self.eps_decay)

def dqn_policy(agent, greedy=False):
    def _fn(env, idle):
        asgn={}
        for ri in idle:
            s=env._state(ri); mask=env.action_mask(); a=agent.act(s,mask,greedy=greedy)
            if mask[a]>0: asgn[ri]=a
        return asgn
    return _fn

def train_dqn(n_train=N_TRAIN, seed_off=0, collision_penalty=-3.0, use_vision=True, use_priority=True, **env_kw):
    env=make_env(planner_mode='vision', use_vision=use_vision, use_priority=use_priority,
                 collision_penalty=collision_penalty, **env_kw)
    ag=DQNAgent(env.state_dim, env.action_dim); rews=[]
    for ep in range(n_train):
        random.seed(SEED+seed_off+ep); np.random.seed(SEED+seed_off+ep); env.reset(); done=False
        while not done:
            idle=[i for i in range(env.N_ROBOTS) if not env.rbusy[i]]
            asgn={}; batch=[]
            for ri in idle:
                s=env._state(ri); mask=env.action_mask(); a=ag.act(s,mask)
                if mask[a]>0: asgn[ri]=a; batch.append((ri,s,a,mask.copy()))
            done=env.step(asgn)
            for ri,s,a,mask in batch:
                ns=env._state(ri); nmask=env.action_mask()
                ag.buf.push(s,a,env.ep_rew/max(1,len(batch)),ns,float(done),mask,nmask)
            ag.learn(); ag.decay()
        rews.append(env.ep_rew)
        if (ep+1)%150==0:
            print(f'  ep {ep+1}/{n_train} eps={ag.eps:.3f} rew={np.mean(rews[-30:]):.1f}')
    return ag, rews

print('DQN defined.')


DQN defined.


In [10]:

# CELL 10 - TRAIN + MULTI-SEED EVALUATION
all_metrics={}; all_train_rews={}; compute_times={}; trained={}

print('Evaluating baselines (multi-seed)...')
for name, fn in POLICIES.items():
    t0=time.time(); all_metrics[name]=eval_policy(name, fn); compute_times[name]=time.time()-t0
    print(f'  {name}: throughput={np.mean(all_metrics[name]["tasks"]):.2f}')

print('Training Q-Learning...')
t0=time.time(); ql_agent, all_train_rews['Q-Learning']=train_ql(400, seed_off=300)
compute_times['Q-Learning']=time.time()-t0
all_metrics['Q-Learning']=eval_policy('Q-Learning', dqn_policy(ql_agent, greedy=True),
    planner_mode='vision', use_vision=True)

print('Training DQN-Vision...')
t0=time.time(); dqn_agent, all_train_rews['DQN-Vision']=train_dqn(N_TRAIN, seed_off=500, use_vision=True, collision_penalty=-3.0)
compute_times['DQN-Vision']=time.time()-t0
all_metrics['DQN-Vision']=eval_policy('DQN-Vision', dqn_policy(dqn_agent, greedy=True),
    planner_mode='vision', use_vision=True)

print('Training Constrained-DQN (stronger collision penalty)...')
t0=time.time(); cdqn_agent, all_train_rews['Constrained-DQN']=train_dqn(N_TRAIN, seed_off=900, use_vision=True, collision_penalty=-8.0)
compute_times['Constrained-DQN']=time.time()-t0
all_metrics['Constrained-DQN']=eval_policy('Constrained-DQN', dqn_policy(cdqn_agent, greedy=True),
    planner_mode='vision', use_vision=True)

trained={'Q-Learning':ql_agent,'DQN-Vision':dqn_agent,'Constrained-DQN':cdqn_agent}

print('\nSUMMARY (mean throughput | collisions | fairness):')
for a in ALGOS:
    m=all_metrics[a]
    print(f'  {a:16s} {np.mean(m["tasks"]):6.2f}  {np.mean(m["colls"]):6.2f}  {np.mean(m["fairness"]):.3f}')


Evaluating baselines (multi-seed)...
  Random: throughput=19.05
  Nearest-WMS: throughput=25.47
  Hungarian-WMS: throughput=26.51
  Genetic-WMS: throughput=26.03
Training Q-Learning...
Training DQN-Vision...
  ep 150/600 eps=0.050 rew=78.5
  ep 300/600 eps=0.050 rew=78.0
  ep 450/600 eps=0.050 rew=101.7
  ep 600/600 eps=0.050 rew=49.3
Training Constrained-DQN (stronger collision penalty)...
  ep 150/600 eps=0.050 rew=-204.4
  ep 300/600 eps=0.050 rew=-98.7
  ep 450/600 eps=0.050 rew=-83.7
  ep 600/600 eps=0.050 rew=-200.8

SUMMARY (mean throughput | collisions | fairness):
  Random            19.05   43.51  0.929
  Nearest-WMS       25.47   30.21  0.910
  Hungarian-WMS     26.51   32.05  0.914
  Genetic-WMS       26.03   33.90  0.926
  Q-Learning        18.84   38.30  0.931
  DQN-Vision        20.01   40.45  0.922
  Constrained-DQN   19.48   41.73  0.924


In [11]:

# CELL 11 - FIG 3 TRAINING CURVES
fig,axes=plt.subplots(1,3,figsize=(16,5))
fig.suptitle('Fig. 3: Training Convergence Curves',fontsize=13,fontweight='bold')
for ax,key,title,col in zip(axes,['Q-Learning','DQN-Vision','Constrained-DQN'],
    ['Q-Learning','DQN-Vision','Constrained-DQN'],['#f39c12','#9b59b6','#8e44ad']):
    r=np.array(all_train_rews[key]); ax.plot(r,color=col+'30',lw=0.6,alpha=0.4)
    ax.plot(smooth(r,25),color=col,lw=2.2,label=title); ax.set_xlabel('Episode'); ax.set_ylabel('Reward')
    ax.set_title(title,fontweight='bold'); ax.legend()
if trained['DQN-Vision'].losses:
    pass
save_fig('fig_03_training_curves'); print('Figure 3 saved.')


  Saved figure: /kaggle/working/figures/fig_03_training_curves.pdf
Figure 3 saved.


In [12]:

# CELL 12 - FIG 4 ALGORITHM COMPARISON
metrics_def=[('tasks','Throughput',True),('ct','Completion Time',False),('dist','Distance',False),
             ('colls','Collisions',False),('fairness','Priority Fairness',True)]
fig,axes=plt.subplots(1,5,figsize=(20,5.5))
fig.suptitle('Fig. 4: Algorithm Comparison Across Logistics KPIs',fontsize=13,fontweight='bold')
for ax,(k,lab,hib) in zip(axes,metrics_def):
    means=[np.mean(all_metrics[a][k]) for a in ALGOS]
    ci=[1.96*np.std(all_metrics[a][k])/np.sqrt(len(all_metrics[a][k])) for a in ALGOS]
    bars=ax.bar(range(len(ALGOS)),means,color=[ALGO_COLORS[a] for a in ALGOS],edgecolor='k',alpha=0.85)
    ax.errorbar(range(len(ALGOS)),means,yerr=ci,fmt='none',color='k',capsize=3)
    best=int(np.argmax(means) if hib else np.argmin(means)); bars[best].set_edgecolor('gold'); bars[best].set_linewidth(2.5)
    ax.set_xticks(range(len(ALGOS))); ax.set_xticklabels([a.replace('-','\n') for a in ALGOS],fontsize=7)
    ax.set_title(lab,fontweight='bold')
save_fig('fig_04_algorithm_comparison'); print('Figure 4 saved.')


  Saved figure: /kaggle/working/figures/fig_04_algorithm_comparison.pdf
Figure 4 saved.


In [13]:

# CELL 13 - FIG 5 VIOLIN + CUMULATIVE
fig,axes=plt.subplots(1,2,figsize=(14,5.5))
fig.suptitle('Fig. 5: Throughput Distribution and Cumulative Performance',fontsize=13,fontweight='bold')
data=[all_metrics[a]['tasks'] for a in ALGOS]
parts=axes[0].violinplot(data,positions=range(len(ALGOS)),showmeans=True,showmedians=True)
for b,c in zip(parts['bodies'],[ALGO_COLORS[a] for a in ALGOS]): b.set_facecolor(c); b.set_alpha(0.75)
axes[0].set_xticks(range(len(ALGOS))); axes[0].set_xticklabels(ALGOS,rotation=20,ha='right')
axes[0].set_ylabel('Tasks / Episode'); axes[0].set_title('(A) Throughput Distribution',fontweight='bold')
for a in ALGOS:
    cs=np.cumsum(all_metrics[a]['tasks'])
    axes[1].plot(cs,color=ALGO_COLORS[a],lw=2,marker=ALGO_MARKERS[a],markevery=15,ms=4,label=a)
axes[1].set_xlabel('Evaluation Episode'); axes[1].set_ylabel('Cumulative Tasks'); axes[1].legend(fontsize=8)
axes[1].set_title('(B) Cumulative Throughput',fontweight='bold')
save_fig('fig_05_distributions_cumulative'); print('Figure 5 saved.')


  Saved figure: /kaggle/working/figures/fig_05_distributions_cumulative.pdf
Figure 5 saved.


In [14]:

# CELL 14 - NOISE ROBUSTNESS (shared perception noise for ALL methods)
NOISE_LEVELS=[0.0,0.05,0.10,0.15,0.25,0.35]
NOISE_ALGOS=['DQN-Vision','Constrained-DQN','Nearest-WMS','Hungarian-WMS','Genetic-WMS']
NOISE_RESULTS={a:defaultdict(list) for a in NOISE_ALGOS}
N_NOISE=40

def eval_at_noise(noise, policy_fn, agent=None, n=N_NOISE):
    m=defaultdict(list)
    for ep in range(n):
        sd=SEED+5000+ep
        env=make_env(noise_sigma=noise, planner_mode='wms', use_vision=False)
        if agent is None:
            met=run_episode(env, policy_fn, sd)
        else:
            env=make_env(noise_sigma=noise, planner_mode='vision', use_vision=True)
            met=run_episode(env, dqn_policy(agent, greedy=True), sd)
        for k,v in met.items(): m[k].append(v)
    return {k:float(np.mean(v)) for k,v in m.items()}

print('Noise robustness (shared task-position + occupancy noise):')
for nv in NOISE_LEVELS:
    print(f'  sigma={nv:.2f}', end='  ')
    NOISE_RESULTS['Nearest-WMS']['tasks'].append(eval_at_noise(nv, policy_nearest_wms)['tasks'])
    NOISE_RESULTS['Nearest-WMS']['colls'].append(eval_at_noise(nv, policy_nearest_wms)['colls'])
    NOISE_RESULTS['Hungarian-WMS']['tasks'].append(eval_at_noise(nv, policy_hungarian_wms)['tasks'])
    NOISE_RESULTS['Hungarian-WMS']['colls'].append(eval_at_noise(nv, policy_hungarian_wms)['colls'])
    NOISE_RESULTS['Genetic-WMS']['tasks'].append(eval_at_noise(nv, policy_genetic_wms)['tasks'])
    NOISE_RESULTS['Genetic-WMS']['colls'].append(eval_at_noise(nv, policy_genetic_wms)['colls'])
    d=eval_at_noise(nv, None, agent=dqn_agent)
    c=eval_at_noise(nv, None, agent=cdqn_agent)
    NOISE_RESULTS['DQN-Vision']['tasks'].append(d['tasks']); NOISE_RESULTS['DQN-Vision']['colls'].append(d['colls'])
    NOISE_RESULTS['Constrained-DQN']['tasks'].append(c['tasks']); NOISE_RESULTS['Constrained-DQN']['colls'].append(c['colls'])
    print(f"DQN={d['tasks']:.2f} NT={NOISE_RESULTS['Nearest-WMS']['tasks'][-1]:.2f}")
print('Noise experiment complete.')


Noise robustness (shared task-position + occupancy noise):
  sigma=0.00  DQN=16.82 NT=27.05
  sigma=0.05  DQN=18.12 NT=27.05
  sigma=0.10  DQN=17.93 NT=26.20
  sigma=0.15  DQN=18.30 NT=25.65
  sigma=0.25  DQN=18.43 NT=25.73
  sigma=0.35  DQN=18.75 NT=27.68
Noise experiment complete.


In [15]:

# CELL 15 - FIG 6 NOISE ROBUSTNESS
fig,axes=plt.subplots(2,2,figsize=(14,10))
fig.suptitle('Fig. 6: Robustness to Shared Perception Noise',fontsize=13,fontweight='bold')
panels=[('tasks','Throughput',True,axes[0,0]),('colls','Collisions',False,axes[0,1])]
for key,ylabel,hib,ax in panels:
    for a in NOISE_ALGOS:
        ax.plot(NOISE_LEVELS,NOISE_RESULTS[a][key],color=ALGO_COLORS[a],marker=ALGO_MARKERS[a],lw=2,ms=5,label=a)
    ax.set_xlabel('Noise sigma'); ax.set_ylabel(ylabel); ax.legend(fontsize=8); ax.set_title(ylabel,fontweight='bold')
ax=axes[1,0]
for a in NOISE_ALGOS:
    base=NOISE_RESULTS[a]['tasks'][0]
    drop=[(base-v)/max(base,1e-9)*100 for v in NOISE_RESULTS[a]['tasks']]
    ax.plot(NOISE_LEVELS,drop,color=ALGO_COLORS[a],marker=ALGO_MARKERS[a],lw=2,label=a)
ax.set_xlabel('Noise sigma'); ax.set_ylabel('Throughput drop (%)'); ax.set_title('Relative Throughput Degradation',fontweight='bold'); ax.legend(fontsize=8)
axes[1,1].axis('off')
axes[1,1].text(0.05,0.95,'Note: All allocators use the same noisy task-position estimates.\nVision-based RL additionally receives noisy occupancy maps.',
               transform=axes[1,1].transAxes,va='top',fontsize=10)
save_fig('fig_06_noise_robustness'); print('Figure 6 saved.')


  Saved figure: /kaggle/working/figures/fig_06_noise_robustness.pdf
Figure 6 saved.


In [16]:

# CELL 16 - ABLATION (vision + priority under dynamic congestion)
ABLATION_CFGS={
    'Full DQN (V+P)':dict(use_vision=True,use_priority=True,collision_penalty=-3.0),
    'No Vision':dict(use_vision=False,use_priority=True,collision_penalty=-3.0),
    'No Priority':dict(use_vision=True,use_priority=False,collision_penalty=-3.0),
    'Constrained Only':dict(use_vision=True,use_priority=True,collision_penalty=-8.0),
}
ablation_metrics={}
print('Ablation study (200 train eps each):')
for name,cfg in ABLATION_CFGS.items():
    off=2000+abs(hash(name))%500
    ag, _ = train_dqn(n_train=200, seed_off=off, **cfg)
    ablation_metrics[name]=eval_policy(name, dqn_policy(ag, greedy=True),
        seeds=[42,123], n_eps=30, planner_mode='vision', use_vision=cfg['use_vision'])
    print(f'  {name}: tasks={np.mean(ablation_metrics[name]["tasks"]):.2f} colls={np.mean(ablation_metrics[name]["colls"]):.2f}')


Ablation study (200 train eps each):
  ep 150/200 eps=0.050 rew=80.9
  Full DQN (V+P): tasks=17.82 colls=44.97
  ep 150/200 eps=0.050 rew=80.3
  No Vision: tasks=16.23 colls=49.55
  ep 150/200 eps=0.050 rew=4.6
  No Priority: tasks=17.02 colls=39.97
  ep 150/200 eps=0.050 rew=-220.7
  Constrained Only: tasks=18.92 colls=31.72


In [20]:

# CELL 17 - FIG 7 ABLATION
abl = list(ABLATION_CFGS.keys())
cols = ['#9b59b6', '#3498db', '#e74c3c', '#8e44ad']

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle(
    'Fig. 7: Ablation - Vision, Priority, and Safety Penalty',
    fontsize=13,
    fontweight='bold'
)

for ax, key in zip(
    axes,
    [('tasks', 'Throughput', True),
     ('colls', 'Collisions', False),
     ('fairness', 'Fairness', True)]
):
    k, lab, hib = key
    means = [np.mean(ablation_metrics[n][k]) for n in abl]
    ax.bar(
        range(len(abl)),
        means,
        color=cols,
        edgecolor='k',
        alpha=0.85
    )
    ax.set_xticks(range(len(abl)))
    ax.set_xticklabels([x.replace(' ', '\n') for x in abl], fontsize=8)
    ax.set_title(lab, fontweight='bold')

save_fig('fig_07_ablation_study')
print('Figure 7 saved.')


  Saved figure: /kaggle/working/figures/fig_07_ablation_study.pdf
Figure 7 saved.


In [21]:

# CELL 18 - FIG 8 RADAR
rdr=['tasks','ct','colls','dist','fairness']; labels=['Throughput','1/ComplTime','1/Collisions','1/Distance','Fairness']
raw={a:np.array([np.mean(all_metrics[a][k]) for k in rdr]) for a in ALGOS}
# invert lower-is-better
for i,k in enumerate(rdr):
    if k in ('ct','colls','dist'): raw={a:np.array(list(v)) for a,v in raw.items()}
for a in ALGOS:
    v=raw[a].copy(); v[1]=1/(v[1]+1); v[2]=1/(v[2]+1); v[3]=1/(v[3]+1); raw[a]=v
norm={}
for i in range(len(rdr)):
    vals=[raw[a][i] for a in ALGOS]; mn,mx=min(vals),max(vals); span=mx-mn if mx!=mn else 1
    for a in ALGOS:
        norm.setdefault(a,np.zeros(len(rdr))); norm[a][i]=(raw[a][i]-mn)/span
fig=plt.figure(figsize=(9,8)); ax=fig.add_subplot(111,polar=True)
fig.suptitle('Fig. 8: Multi-Metric Radar (normalized)',fontsize=13,fontweight='bold')
N=len(labels); ang=np.linspace(0,2*np.pi,N,endpoint=False).tolist(); ang+=ang[:1]
for a in ALGOS:
    v=list(norm[a])+[norm[a][0]]; ax.plot(ang,v,'o-',lw=1.8,color=ALGO_COLORS[a],label=a); ax.fill(ang,v,alpha=0.12,color=ALGO_COLORS[a])
ax.set_xticks(ang[:-1]); ax.set_xticklabels(labels,fontsize=9); ax.legend(loc='upper right',bbox_to_anchor=(1.35,1.15),fontsize=8)
save_fig('fig_08_radar_chart'); print('Figure 8 saved.')


  Saved figure: /kaggle/working/figures/fig_08_radar_chart.pdf
Figure 8 saved.


In [22]:

# CELL 19 - FIG 9 PARETO
fig,axes=plt.subplots(1,2,figsize=(14,6))
fig.suptitle('Fig. 9: Safety-Energy-Throughput Trade-offs',fontsize=13,fontweight='bold')
pairs=[('batt','Battery Use','tasks','Throughput','(A) Energy-Throughput'),('colls','Collisions','tasks','Throughput','(B) Safety-Throughput')]
for ax,(xk,xl,yk,yl,ttl) in zip(axes,pairs):
    for a in ALGOS:
        xs=np.array(all_metrics[a][xk]); ys=np.array(all_metrics[a][yk])
        ax.scatter(xs,ys,c=ALGO_COLORS[a],alpha=0.15,s=15)
        ax.scatter(np.mean(xs),np.mean(ys),c=ALGO_COLORS[a],s=140,marker=ALGO_MARKERS[a],edgecolors='k',label=a)
    ax.set_xlabel(xl); ax.set_ylabel(yl); ax.set_title(ttl,fontweight='bold'); ax.legend(fontsize=7,ncol=2)
save_fig('fig_09_pareto_tradeoff'); print('Figure 9 saved.')


  Saved figure: /kaggle/working/figures/fig_09_pareto_tradeoff.pdf
Figure 9 saved.


In [23]:

# CELL 20 - FIG 10 BOXPLOTS
fig,axes=plt.subplots(2,3,figsize=(17,10))
fig.suptitle('Fig. 10: Distribution Comparison Across Evaluation Episodes',fontsize=13,fontweight='bold')
cfg=[('tasks','Throughput'),('ct','Compl. Time'),('dist','Distance'),('colls','Collisions'),('fairness','Fairness'),('reward','Reward')]
for ax,(k,lab) in zip(axes.ravel(),cfg):
    data=[all_metrics[a][k] for a in ALGOS]
    bp=ax.boxplot(data,patch_artist=True,notch=True)
    for patch,c in zip(bp['boxes'],[ALGO_COLORS[a] for a in ALGOS]): patch.set_facecolor(c); patch.set_alpha(0.75)
    ax.set_xticklabels(ALGOS,rotation=20,ha='right',fontsize=7); ax.set_title(lab,fontweight='bold')
save_fig('fig_10_statistical_comparison'); print('Figure 10 saved.')


  Saved figure: /kaggle/working/figures/fig_10_statistical_comparison.pdf
Figure 10 saved.


In [24]:

# CELL 21 - FIG 11 HEATMAPS
env_hm=make_env(use_vision=True); rmaps=[np.zeros((8,8)) for _ in range(3)]; tmap=np.zeros((8,8))
pol=dqn_policy(dqn_agent, greedy=True)
for ep in range(25):
    random.seed(SEED+7000+ep); env_hm.reset(); done=False
    while not done:
        for i in range(3): r,c=env_hm.rpos[i]; rmaps[i][r,c]+=1
        for j in range(6):
            if env_hm.tactive[j]: r,c=env_hm.tpos[j]; tmap[r,c]+=1
        idle=[i for i in range(3) if not env_hm.rbusy[i]]
        done=env_hm.step(pol(env_hm, idle))
fig,axes=plt.subplots(2,2,figsize=(13,11))
fig.suptitle('Fig. 11: DQN-Vision Spatial Activity Heatmaps',fontsize=13,fontweight='bold')
for i,ax in enumerate(axes.flat[:3]):
    im=ax.imshow(np.ma.masked_where(env_hm.shelf==1,rmaps[i]),cmap=['Blues','Purples','Greens'][i])
    plt.colorbar(im,ax=ax,fraction=0.046); ax.set_title(f'Robot R{i+1}'); ax.invert_yaxis()
im=axes[1,1].imshow(np.ma.masked_where(env_hm.shelf==1,tmap),cmap='YlOrRd')
plt.colorbar(im,ax=axes[1,1],fraction=0.046); axes[1,1].set_title('Task Frequency'); axes[1,1].invert_yaxis()
save_fig('fig_11_heatmaps'); del env_hm; print('Figure 11 saved.')


  Saved figure: /kaggle/working/figures/fig_11_heatmaps.pdf
Figure 11 saved.


In [25]:

# CELL 22 - FIG 12 LEARNING EFFICIENCY
fig,axes=plt.subplots(1,2,figsize=(14,5))
fig.suptitle('Fig. 12: Training Efficiency and Compute Cost',fontsize=13,fontweight='bold')
for key,col in [('Q-Learning','#f39c12'),('DQN-Vision','#9b59b6'),('Constrained-DQN','#8e44ad')]:
    r=smooth(np.array(all_train_rews[key]),25); axes[0].plot(r,color=col,lw=2,label=key)
axes[0].set_xlabel('Episode'); axes[0].set_ylabel('Smoothed Reward'); axes[0].legend(); axes[0].set_title('Training Curves',fontweight='bold')
ct=[compute_times[a] for a in ALGOS]; tp=[np.mean(all_metrics[a]['tasks']) for a in ALGOS]
axes[1].scatter(ct,tp,s=[120]*len(ALGOS),c=[ALGO_COLORS[a] for a in ALGOS],edgecolors='k')
for a,c,t in zip(ALGOS,ct,tp): axes[1].annotate(a,(c,t),fontsize=8,xytext=(4,4),textcoords='offset points')
axes[1].set_xlabel('Compute time (s)'); axes[1].set_ylabel('Throughput'); axes[1].set_title('Compute vs Throughput',fontweight='bold')
save_fig('fig_12_learning_efficiency'); print('Figure 12 saved.')


  Saved figure: /kaggle/working/figures/fig_12_learning_efficiency.pdf
Figure 12 saved.


In [26]:

# CELL 23 - SAVE ALL TABLES
# Table 2 main comparison
rows=[]
for a in ALGOS:
    row={'Algorithm':a,'Data_Source':DATA_SOURCE}
    for k in ['tasks','ct','dist','colls','batt','fairness','reward']:
        v=np.array(all_metrics[a][k])
        row[f'{k}_mean']=round(float(np.mean(v)),4); row[f'{k}_std']=round(float(np.std(v)),4)
    rows.append(row)
df_t2=pd.DataFrame(rows); df_t2.to_csv(f'{TAB_DIR}/table_02_algorithm_comparison.csv',index=False)

# Table 3 noise
rows=[]
for i,nv in enumerate(NOISE_LEVELS):
    for a in NOISE_ALGOS:
        rows.append({'Noise_Sigma':nv,'Algorithm':a,'Throughput':NOISE_RESULTS[a]['tasks'][i],'Collisions':NOISE_RESULTS[a]['colls'][i]})
df_t3=pd.DataFrame(rows); df_t3.to_csv(f'{TAB_DIR}/table_03_noise_robustness.csv',index=False)

# Table 4 ablation
rows=[]
for n in ablation_metrics:
    m=ablation_metrics[n]
    rows.append({'Variant':n,'Throughput':np.mean(m['tasks']),'Collisions':np.mean(m['colls']),'Fairness':np.mean(m['fairness'])})
df_t4=pd.DataFrame(rows); df_t4.to_csv(f'{TAB_DIR}/table_04_ablation_study.csv',index=False)

# Table 5 compute
df_t5=pd.DataFrame([{'Algorithm':a,'Compute_sec':round(compute_times[a],2),'Throughput':round(np.mean(all_metrics[a]['tasks']),3)} for a in ALGOS])
df_t5.to_csv(f'{TAB_DIR}/table_05_computational_metrics.csv',index=False)

# Table 6 stats vs DQN-Vision
dqn_t=np.array(all_metrics['DQN-Vision']['tasks']); rows=[]
for a in ALGOS:
    if a=='DQN-Vision': continue
    o=np.array(all_metrics[a]['tasks'])
    stat,p=mannwhitneyu(dqn_t,o,alternative='two-sided')
    pooled=np.sqrt((np.var(dqn_t)+np.var(o))/2)+1e-9
    d=(np.mean(dqn_t)-np.mean(o))/pooled
    sig='***' if p<0.001 else '**' if p<0.01 else '*' if p<0.05 else 'ns'
    rows.append({'Comparison':f'DQN-Vision vs {a}','P_Value':round(float(p),6),'Significance':sig,'Cohens_d':round(float(d),4)})
df_t6=pd.DataFrame(rows); df_t6.to_csv(f'{TAB_DIR}/table_06_statistical_tests.csv',index=False)

# Table 7 multi-seed summary
rows=[]
for a in ALGOS:
    for sd in EVAL_SEEDS:
        env=make_env(planner_mode='wms')
        if a in POLICIES:
            fn=POLICIES[a]
            vals=[run_episode(env,fn,sd*1000+ep)['tasks'] for ep in range(20)]
        elif a=='Q-Learning':
            vals=[run_episode(env,dqn_policy(ql_agent,True),sd*1000+ep)['tasks'] for ep in range(20)]
        elif a=='DQN-Vision':
            vals=[run_episode(env,dqn_policy(dqn_agent,True),sd*1000+ep)['tasks'] for ep in range(20)]
        else:
            vals=[run_episode(env,dqn_policy(cdqn_agent,True),sd*1000+ep)['tasks'] for ep in range(20)]
        rows.append({'Algorithm':a,'Seed':sd,'Throughput_Mean':round(float(np.mean(vals)),3)})
df_t7=pd.DataFrame(rows); df_t7.to_csv(f'{TAB_DIR}/table_07_multiseed_summary.csv',index=False)

print('All tables saved to', TAB_DIR)
print(df_t2[['Algorithm','tasks_mean','colls_mean','fairness_mean']].to_string(index=False))


All tables saved to /kaggle/working/tables
      Algorithm  tasks_mean  colls_mean  fairness_mean
         Random     19.0533     43.5067         0.9287
    Nearest-WMS     25.4733     30.2133         0.9098
  Hungarian-WMS     26.5133     32.0467         0.9138
    Genetic-WMS     26.0333     33.9000         0.9263
     Q-Learning     18.8400     38.3000         0.9305
     DQN-Vision     20.0067     40.4533         0.9223
Constrained-DQN     19.4800     41.7333         0.9237


In [27]:

# CELL 24 - ZIP OUTPUTS
ZIP_PATH=f'{OUT}/vision_warehouse_rl_outputs.zip'
with zipfile.ZipFile(ZIP_PATH,'w',zipfile.ZIP_DEFLATED) as zf:
    for fp in sorted(glob.glob(f'{FIG_DIR}/*.pdf')): zf.write(fp,f'figures/{os.path.basename(fp)}')
    for fp in sorted(glob.glob(f'{TAB_DIR}/*.csv')): zf.write(fp,f'tables/{os.path.basename(fp)}')
print('ZIP:', ZIP_PATH, os.path.getsize(ZIP_PATH)//1024, 'KB')
print('Figures:', len(glob.glob(f'{FIG_DIR}/*.pdf')), '| Tables:', len(glob.glob(f'{TAB_DIR}/*.csv')))


ZIP: /kaggle/working/vision_warehouse_rl_outputs.zip 350 KB
Figures: 12 | Tables: 7


In [28]:

# CELL 25 - FINAL SUMMARY
print('='*72)
print('FINAL RESULTS | Data source:', DATA_SOURCE)
print('='*72)
print(f"{'Algorithm':16s} {'Throughput':>11s} {'Collisions':>11s} {'Fairness':>9s} {'ComplT':>8s}")
for a in ALGOS:
    m=all_metrics[a]
    print(f"{a:16s} {np.mean(m['tasks']):9.2f}   {np.mean(m['colls']):9.2f}   {np.mean(m['fairness']):7.3f}   {np.mean(m['ct']):7.2f}")
best_t=ALGOS[int(np.argmax([np.mean(all_metrics[x]['tasks']) for x in ALGOS]))]
best_s=ALGOS[int(np.argmin([np.mean(all_metrics[x]['colls']) for x in ALGOS]))]
print('\nBest throughput :', best_t)
print('Fewest collisions:', best_s)
print('Download:', f'{OUT}/vision_warehouse_rl_outputs.zip')
print('='*72)


FINAL RESULTS | Data source: kaggle:Updated_Warehouse_Operations_Dataset.csv
Algorithm         Throughput  Collisions  Fairness   ComplT
Random               19.05       43.51     0.929     13.10
Nearest-WMS          25.47       30.21     0.910      9.87
Hungarian-WMS        26.51       32.05     0.914      9.46
Genetic-WMS          26.03       33.90     0.926      9.28
Q-Learning           18.84       38.30     0.931     12.72
DQN-Vision           20.01       40.45     0.922     13.33
Constrained-DQN      19.48       41.73     0.924     12.83

Best throughput : Hungarian-WMS
Fewest collisions: Nearest-WMS
Download: /kaggle/working/vision_warehouse_rl_outputs.zip
